# Faruq-v3 — AF2 + CPE0 seed-42 matched screen

Pola eksekusi mengikuti notebook eksperimen terbaru: repo di-fetch/reset ke branch frozen, package di-reload, satu worker menjalankan static audit → matched control `AF2CPE0` → candidate `AF2CPE5` → frozen decision. Stdout/stderr worker ditulis ke log dan tail log dicetak otomatis jika worker gagal. Validation-only; test tidak boleh tersedia.


In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import json, os, shutil, subprocess, sys, tarfile, time
from pathlib import Path

REPO = Path('/content/coffee-bean-detection')
BRANCH = 'codex/af2-cpe0-seed42'
if (REPO / '.git').is_dir():
    subprocess.run(['git', 'fetch', 'origin', BRANCH], cwd=REPO, check=True)
    subprocess.run(['git', 'checkout', BRANCH], cwd=REPO, check=True)
    subprocess.run(['git', 'reset', '--hard', f'origin/{BRANCH}'], cwd=REPO, check=True)
else:
    if REPO.exists():
        shutil.rmtree(REPO)
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', BRANCH, 'https://github.com/ediprin/coffee-bean-detection.git', str(REPO)], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(REPO)], check=True)
for key in list(sys.modules):
    if key == 'coffee_detector' or key.startswith('coffee_detector.'):
        sys.modules.pop(key, None)
sys.path.insert(0, str(REPO / 'src'))
os.chdir(REPO)
print('COMMIT:', subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=REPO, text=True).strip())


In [ ]:
import torch
from coffee_detector.drive_project import require_project_artifact, resolve_drive_project_root

assert torch.cuda.is_available(), 'Aktifkan GPU pada runtime ini.'
REQUIRED = (
    'bundles/faruq-development-v3-grouped.tar',
    'experiments/faruq-v3-breadth-screening-batch-v1/candidates/AFAB/AF2_seed42/weights/best.pt',
)
PROJECT_ROOT = resolve_drive_project_root(required_relative_paths=REQUIRED)
ARCHIVE = require_project_artifact(PROJECT_ROOT, REQUIRED[0])
AF2_CHECKPOINT = require_project_artifact(PROJECT_ROOT, REQUIRED[1])
DATA_ROOT = Path('/content/faruq-development-v3-grouped')
if not (DATA_ROOT / 'data.yaml').is_file():
    with tarfile.open(ARCHIVE, 'r') as archive:
        archive.extractall('/content', filter='data')
GROUPED_SUMMARY = DATA_ROOT / 'faruq_grouped_summary.json'
assert GROUPED_SUMMARY.is_file(), GROUPED_SUMMARY
assert not (DATA_ROOT / 'test').exists(), 'Test tidak boleh tersedia.'
OUTPUT_ROOT = PROJECT_ROOT / 'experiments/faruq-v3-af2-cpe0-seed42-v1'
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
print('GPU       :', torch.cuda.get_device_name(0))
print('AF2       :', AF2_CHECKPOINT)
print('DATA      :', DATA_ROOT)
print('OUTPUTROOT:', OUTPUT_ROOT)


In [ ]:
# Satu worker, resume-safe. Jika gagal, traceback asli otomatis ditampilkan dari log.
command = [
    sys.executable, '-u', '-m', 'coffee_detector.experiments.run_faruq_v3_af2_cpe_seed42_worker',
    '--data-root', str(DATA_ROOT),
    '--grouped-summary', str(GROUPED_SUMMARY),
    '--af2-checkpoint', str(AF2_CHECKPOINT),
    '--output-root', str(OUTPUT_ROOT),
    '--seed', '42', '--device', '0', '--authorize-training',
]
RUN_LOG = Path('/content/af2_cpe0_seed42.log')
print('MENJALANKAN:', ' '.join(command), flush=True)
with RUN_LOG.open('w', encoding='utf-8', buffering=1) as log_stream:
    process = subprocess.Popen(command, cwd=REPO, text=True, stdout=log_stream, stderr=subprocess.STDOUT)
    while process.poll() is None:
        progress = []
        for arm in ('AF2CPE0', 'AF2CPE5'):
            csv_path = OUTPUT_ROOT / arm / f'{arm}_seed42' / 'results.csv'
            epochs = 0
            if csv_path.is_file():
                try:
                    import pandas as pd
                    epochs = len(pd.read_csv(csv_path))
                except Exception:
                    pass
            progress.append(f'{arm}={epochs}/50')
        audit_state = 'ready' if (OUTPUT_ROOT / 'static_audit.json').is_file() else 'running'
        print(f'[SEED 42] static={audit_state} | ' + ' | '.join(progress), flush=True)
        time.sleep(60)
    return_code = process.wait()
if return_code != 0:
    tail = '\n'.join(RUN_LOG.read_text(encoding='utf-8', errors='replace').splitlines()[-220:])
    print('===== WORKER LOG TAIL =====')
    print(tail)
    raise RuntimeError(f'AF2+CPE0 seed42 worker gagal; log={RUN_LOG}')
print('WORKER SELESAI')


In [ ]:
# Ringkasan hasil. Cell ini aman direrun setelah worker selesai.
import pandas as pd
from IPython.display import display
WORKER = OUTPUT_ROOT / 'val_reports' / 'af2_cpe_seed42_worker.json'
DECISION = OUTPUT_ROOT / 'decision_seed42.json'
assert WORKER.is_file(), WORKER
assert DECISION.is_file(), DECISION
worker = json.loads(WORKER.read_text(encoding='utf-8'))
rows = []
for arm in ('AF2CPE0', 'AF2CPE5'):
    m = worker[arm]
    rows.append({
        'arm': arm,
        'Macro mAP50-95': m['macro_map50_95'],
        'Bottom-3': m['bottom3_class_map50_95'],
        'Worst': m['worst_class_map50_95'],
    })
display(pd.DataFrame(rows).style.format({
    'Macro mAP50-95': '{:.2%}',
    'Bottom-3': '{:.2%}',
    'Worst': '{:.2%}',
}))
decision = json.loads(DECISION.read_text(encoding='utf-8'))
print(json.dumps(decision, indent=2))
print('DECISION:', decision['decision'])
print('LOG     :', RUN_LOG)
print('SUMMARY :', WORKER)
